## Installing

In [ ]:
!git clone https://github.com/DependableSystemsLab/SolidiFI-benchmark.git
!git clone https://github.com/smartbugs/smartbugs-curated.git
!git clone https://github.com/wuhongjun15/Peculiar.git
!git clone https://github.com/MRdoulestar/DeeSCVHunter.git
!git clone https://github.com/smartbugs/smartbugs-wild.git

Cloning into 'SolidiFI-benchmark'...
remote: Enumerating objects: 2690, done.
remote: Counting objects: 100% (83/83), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 2690 (delta 27), reused 81 (delta 26), pack-reused 2607 (from 1)
Receiving objects: 100% (2690/2690), 7.29 MiB | 8.38 MiB/s, done.
Resolving deltas: 100% (2134/2134), done.
Updating files: 100% (5107/5107), done.
Cloning into 'smartbugs-curated'...
remote: Enumerating objects: 225, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 225 (delta 29), reused 19 (delta 19), pack-reused 177 (from 1)
Receiving objects: 100% (225/225), 173.89 KiB | 917.00 KiB/s, done.
Resolving deltas: 100% (65/65), done.
Cloning into 'Peculiar'...
remote: Enumerating objects: 85, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 85 (delta 2), reused 11 (delta 2), pack-reused 74 (from 1)
Receiving obje

In [ ]:
import re
import random
random.seed(42)

# Keywords of Solidity; immutable set
keywords = frozenset({
    'bool', 'break', 'case', 'catch', 'const', 'continue', 'default', 'do', 'double', 'struct',
    'else', 'enum', 'payable', 'function', 'modifier', 'emit', 'export', 'extern', 'false', 'constructor',
    'float', 'if', 'contract', 'int', 'long', 'string', 'super', 'or', 'private', 'protected', 'noReentrancy',
    'public', 'return', 'returns', 'assert', 'event', 'indexed', 'using', 'require', 'uint', 'onlyDaoChallenge',
    'transfer', 'Transfer', 'Transaction', 'switch', 'pure', 'view', 'this', 'throw', 'true', 'try', 'revert',
    'bytes', 'bytes4', 'bytes32', 'internal', 'external', 'union', 'constant', 'while', 'for', 'notExecuted',
    'NULL', 'uint256', 'uint128', 'uint8', 'uint16', 'address', 'call', 'msg', 'value', 'sender', 'notConfirmed',
    'private', 'onlyOwner', 'internal', 'onlyGovernor', 'onlyCommittee', 'onlyAdmin', 'onlyPlayers', 'ownerExists',
    'onlyManager', 'onlyHuman', 'only_owner', 'onlyCongressMembers', 'preventReentry', 'noEther', 'onlyMembers',
    'onlyProxyOwner', 'confirmed', 'mapping', 'solidity'
})

global_vars = frozenset({
    'block.timestamp', 'now', 'msg.sender', 'msg.value', 'block.number', 'block.difficulty',
    'block.coinbase', 'block.gaslimit', 'tx.origin', 'tx.gasprice', 'gasleft', 'this', 'super'
})

# Known non-user-defined functions; immutable set
main_set = frozenset({'function', 'constructor', 'modifier', 'contract'})
main_args = frozenset({'argc', 'argv'})

def clean_fragment(fragment):
    fun_symbols = {}
    var_symbols = {}
    fun_count = 1
    var_count = 1

    rx_fun = re.compile(r'\b([_A-Za-z]\w*)\b(?=\s*\()')
    rx_var = re.compile(r'\b([_A-Za-z]\w*)\b(?:(?=\s*\w+\()|(?!\s*\w+))(?!\s*\()')

    cleaned_fragment = []

    for line in fragment:
        # Skip lines that are comments (single-line or multi-line)
        if line.strip().startswith('//') or line.strip().startswith('/*') or line.strip().endswith('*/'):
            cleaned_fragment.append(line)
            continue

        # Remove string literals and non-ASCII chars from the code part
        nostrlit_line = re.sub(r'".*?"', '""', line)
        nocharlit_line = re.sub(r"'.*?'", "''", nostrlit_line)
        ascii_line = re.sub(r'[^\x00-\x7f]', '', nocharlit_line)

        # Process function and variable names
        user_fun = rx_fun.findall(ascii_line)
        user_var = rx_var.findall(ascii_line)

        for fun_name in user_fun:
            if fun_name not in main_set and fun_name not in keywords:
                if fun_name not in fun_symbols:
                    fun_symbols[fun_name] = f'FUN{fun_count}'
                    fun_count += 1
                ascii_line = re.sub(r'\b' + fun_name + r'\b(?=\s*\()', fun_symbols[fun_name], ascii_line)

        for var_name in user_var:
            if var_name not in keywords and var_name not in main_args and var_name not in global_vars:
                if var_name not in var_symbols:
                    var_symbols[var_name] = f'VAR{var_count}'
                    var_count += 1
                ascii_line = re.sub(r'\b' + var_name + r'\b(?:(?=\s*\w+\()|(?!\s*\w+))(?!\s*\()', var_symbols[var_name], ascii_line)

        cleaned_fragment.append(ascii_line)

    return cleaned_fragment

def remove_comments(solidity_code):
    # Regex patterns to match single-line and multi-line comments
    single_line_comment_pattern = r"//.*?(?=\n|$)"
    multi_line_comment_pattern = r"/\*.*?\*/"

    # Remove single-line comments
    code_without_single_line_comments = re.sub(single_line_comment_pattern, '', solidity_code, flags=re.DOTALL)

    # Remove multi-line comments
    cleaned_code = re.sub(multi_line_comment_pattern, '', code_without_single_line_comments, flags=re.DOTALL)

    return cleaned_code.strip()

def clean_code_formatting(code):
    # Split code into lines
    lines = code.split('\n')

    cleaned_lines = []
    for line in lines:
        # Remove trailing whitespace
        line = line.rstrip()

        # Skip empty lines (preserve newlines with content)
        if not line.strip():
            continue

        # Collapse multiple spaces/tabs (except in string literals)
        line = re.sub(r'(?<!")\s{2,}(?!")', ' ', line)

        cleaned_lines.append(line)

    # Join lines while preserving single newlines
    return '\n'.join(cleaned_lines)

## SolidiFI-Benchmark

In [ ]:
import os

root = "/content/SolidiFI-benchmark/buggy_contracts"
vulnerabilities = os.listdir(root)
solidiFI_files =  {}
for vulnerability in vulnerabilities:
  solidiFI_files[vulnerability] = {}

for vulnerability in vulnerabilities:
    for file in os.listdir(f"{root}/{vulnerability}"):
        if file.endswith(".sol"):
            sol_code = open(f"{root}/{vulnerability}/{file}", "r").readlines()
            sol_code = "\n".join(sol_code)
            file_name = file.split(".")[0]
            solidiFI_files[vulnerability][file_name] = sol_code

In [ ]:
for vulnerability in vulnerabilities:
    tmp = [solidiFI_files[vulnerability][i] for i in solidiFI_files[vulnerability].keys()]
    print(f"Number of source code with {vulnerability}", len(set(tmp)))

Number of source code with Timestamp-Dependency 49
Number of source code with Overflow-Underflow 49
Number of source code with Unchecked-Send 49
Number of source code with Re-entrancy 49
Number of source code with TOD 49
Number of source code with Unhandled-Exceptions 49
Number of source code with tx.origin 49


In [ ]:
print(clean_code_formatting(remove_comments(solidiFI_files["Timestamp-Dependency"]["buggy_21"])))

pragma solidity ^0.5.11;
contract Token {
 function transfer(address to, uint256 value) public returns (bool success);
address winner_tmstmp7;
function play_tmstmp7(uint startTime) public {
	uint _vtime = block.timestamp;
	if (startTime + (5 * 1 days) == _vtime){
 winner_tmstmp7 = msg.sender;}}
 function transferFrom(address from, address to, uint256 value) public returns (bool success);
address winner_tmstmp23;
function play_tmstmp23(uint startTime) public {
	uint _vtime = block.timestamp;
	if (startTime + (5 * 1 days) == _vtime){
 winner_tmstmp23 = msg.sender;}}
 function balanceOf(address account) external view returns(uint256);
address winner_tmstmp14;
function play_tmstmp14(uint startTime) public {
	if (startTime + (5 * 1 days) == block.timestamp){
 winner_tmstmp14 = msg.sender;}}
 function allowance(address _owner, address _spender)external view returns(uint256);
address winner_tmstmp30;
function play_tmstmp30(uint startTime) public {
	if (startTime + (5 * 1 days) == block.timest

In [ ]:
import re
import pandas as pd

def find_function(lines, error_line):
    function_pattern = re.compile(r'\bfunction\s+\w+\s*\([^)]*\)\s*(\{[^}]*\})?')
    modifier_pattern = re.compile(r'\bmodifier\s+\w+\s*\([^)]*\)\s*(\{[^}]*\})?')
    constructor_pattern = re.compile(r'\bconstructor\s*\([^)]*\)\s*(\{[^}]*\})?')
    start_line = None

    # Search backward for the start of the function
    for i in range(error_line - 1, -1, -1):
        if function_pattern.search(lines[i]) or modifier_pattern.search(lines[i]) or constructor_pattern.search(lines[i]):
            start_line = i
            break

    if start_line is None:
        raise Exception("Function definition not found.")

    # Search forward for the end of the function
    end_line = None
    brace_count = 0
    in_function = False

    for i in range(start_line, len(lines)):
        line = lines[i]
        brace_count += line.count('{')
        brace_count -= line.count('}')

        if brace_count > 0:
            in_function = True
        elif brace_count == 0 and in_function:
            end_line = i
            break

    if end_line is None:
        raise Exception("Function end not found.")

    # Extract the function code
    function_code = "".join(lines[start_line:end_line + 1])
    return function_code


solidiFI_funcs =  {}
for vulnerability in vulnerabilities:
    solidiFI_funcs[vulnerability] = {
        "vulnerable": [],
        "non-vulnerable": []
    }


for vulnerability in vulnerabilities:
    for name, file in solidiFI_files[vulnerability].items():
        lines = file.splitlines(keepends=True)
        file_num = name[6:]
        csv_path = f"/content/SolidiFI-benchmark/buggy_contracts/{vulnerability}/BugLog_{file_num}.csv"
        tdf = pd.read_csv(csv_path)
        vul_line_num = list(tdf['loc'])

        vul_funcs = []
        all_funcs = []

        for line in range(len(lines)):
            try:
                func = find_function(lines, line)
                if "contract" not in func:
                    func = remove_comments(func).replace("\n\n\n\n\n", "\n").replace("\n\n\n\n", "\n").replace("\n\n\n", "\n").replace("\n\n", "\n")
                    all_funcs.append(func.strip())
            except:
                pass
        for er_line in vul_line_num:
            try:
                func = find_function(lines, er_line)
                if file_num == "21" and vulnerability == "Timestamp-Dependency":
                    print(func)
                    print("-" * 100)
                if "contract" not in func:
                    func = remove_comments(func).replace("\n\n\n\n\n", "\n").replace("\n\n\n\n", "\n").replace("\n\n\n", "\n").replace("\n\n", "\n")
                    vul_funcs.append(func.strip())
            except:
                pass

        vul_funcs = list(set(vul_funcs))
        all_funcs = list(set(all_funcs) - set(vul_funcs))
        solidiFI_funcs[vulnerability]["vulnerable"].extend(vul_funcs)
        solidiFI_funcs[vulnerability]["non-vulnerable"].extend(all_funcs)

      function mul(uint256 a, uint256 b) internal pure returns (uint256) 

    {

        if (a == 0) {

        return 0;}

        uint256 c = a * b;

        assert(c / a == b);

        return c;

    }

----------------------------------------------------------------------------------------------------
  function bug_tmstmp25() view public returns (bool) {

    return block.timestamp >= 1546300800;

  }

----------------------------------------------------------------------------------------------------
function bug_tmstmp40 () public payable {

	uint pastBlockTime_tmstmp40; // Forces one bet per block

	require(msg.value == 10 ether); // must send 10 ether to play

        require(now != pastBlockTime_tmstmp40); // only 1 transaction per block   //bug

        pastBlockTime_tmstmp40 = now;       //bug

        if(now % 15 == 0) { // winner    //bug

            msg.sender.transfer(address(this).balance);

        }

    }

---------------------------------------------------------

In [ ]:
for vulnerability in vulnerabilities:
  print(f"Number of vulnerable functions with {vulnerability}", len(solidiFI_funcs[vulnerability]["vulnerable"]))
  print(f"Number of non-vulnerable functions with {vulnerability}", len(solidiFI_funcs[vulnerability]["non-vulnerable"]))
  print("-" * 100)

Number of vulnerable functions with Timestamp-Dependency 756
Number of non-vulnerable functions with Timestamp-Dependency 1328
----------------------------------------------------------------------------------------------------
Number of vulnerable functions with Overflow-Underflow 874
Number of non-vulnerable functions with Overflow-Underflow 1716
----------------------------------------------------------------------------------------------------
Number of vulnerable functions with Unchecked-Send 676
Number of non-vulnerable functions with Unchecked-Send 1542
----------------------------------------------------------------------------------------------------
Number of vulnerable functions with Re-entrancy 773
Number of non-vulnerable functions with Re-entrancy 1297
----------------------------------------------------------------------------------------------------
Number of vulnerable functions with TOD 1163
Number of non-vulnerable functions with TOD 2462
----------------------------

In [ ]:
print(clean_code_formatting(solidiFI_funcs["Timestamp-Dependency"]["vulnerable"][20]))
cnt = 0
for function in solidiFI_funcs["Timestamp-Dependency"]["vulnerable"]:
    if "block.timestamp" in function:
        cnt += 1

print(cnt)

function play_tmstmp30(uint startTime) public {
	if (startTime + (5 * 1 days) == block.timestamp){
 winner_tmstmp30 = msg.sender;}}
359


In [ ]:
cnt = 0
for function in solidiFI_funcs["Re-entrancy"]["vulnerable"]:
    if "call.value" in function:
        cnt += 1

print(cnt)

124


## SmartBugs_Curated

In [ ]:
import json

with open("/content/smartbugs-curated/vulnerabilities.json") as f:
    data = json.load(f)

smartbugs_files, smartbugs_functions = {}, {}

for file_dis in data:
    vulnerability = file_dis["path"].split("/")[1]
    file_name = file_dis["path"].split("/")[-1]
    file_name = file_name.split(".")[0]
    if vulnerability not in smartbugs_files:
        smartbugs_files[vulnerability] = {}
        smartbugs_functions[vulnerability] = {
            "vulnerable": [],
            "non-vulnerable": []
        }
    with open(f"/content/smartbugs-curated/{file_dis['path']}", "r") as f:
        code = f.read()
    ok = 0
    lines = code.splitlines(keepends=True)
    code = "\n".join(lines)
    code = remove_comments(code)
    for er_line in file_dis["vulnerabilities"]:
        try:
            func = find_function(lines, er_line["lines"][0])
            if vulnerability == "reentrancy":
                if "call.value" in func:
                    ok = 1
            elif vulnerability == "time_manipulation":
                if "now" in func or "block.timestamp" in func:
                    ok = 1
            if ok == 1:
                smartbugs_functions[vulnerability]["vulnerable"].append(remove_comments(func.strip()))
        except Exception as e:
            print(e)
            pass
    if ok == 1:
        smartbugs_files[vulnerability][code] = file_name

Function definition not found.
Function definition not found.


In [ ]:
for vulnerability in smartbugs_files.keys():
    smartbugs_functions[vulnerability]["vulnerable"] = list(set(smartbugs_functions[vulnerability]["vulnerable"]))
    if vulnerability == "reentrancy" or vulnerability == "time_manipulation":
        print(f"Number of source code with {vulnerability}", len(smartbugs_files[vulnerability]))
        print(f"Number of vulnerable functions with {vulnerability}", len(smartbugs_functions[vulnerability]["vulnerable"]))
        print("-" * 100)

Number of source code with reentrancy 28
Number of vulnerable functions with reentrancy 14
----------------------------------------------------------------------------------------------------
Number of source code with time_manipulation 5
Number of vulnerable functions with time_manipulation 6
----------------------------------------------------------------------------------------------------


## Peculiar Dataset

In [ ]:
!unzip /content/Peculiar/dataset.zip -d /content/

Archive:  /content/Peculiar/dataset.zip
replace /content/data.jsonl? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: /content/data.jsonl     
replace /content/test.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: a
error:  invalid response [a]
replace /content/test.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: /content/test.txt       
  inflating: /content/train.txt      
  inflating: /content/valid.txt      


In [ ]:
import json

def read_jsonl(file_path):
  data = []
  try:
    with open(file_path, 'r') as f:
      for line in f:
        try:
          data.append(json.loads(line))
        except json.JSONDecodeError as e:
          print(f"Skipping invalid JSON line: {line.strip()}. Error: {e}")
    return data
  except FileNotFoundError:
    print(f"Error: File not found at {file_path}")
    return None

# Example usage:
file_path = '/content/data.jsonl' # replace with actual path
data = read_jsonl(file_path)

In [ ]:
def merge_contracts(contracts):
    if not contracts:
        return ""

    pragma_set = set()
    extracted_contracts = []

    # Helper functions to extract contracts and their names
    def extract_contracts(code):
        contracts = []
        current_contract = []
        in_contract = False
        brace_count = 0
        lines = code.split('\n')
        for line in lines:
            stripped_line = line.strip()
            if not in_contract:
                if stripped_line.startswith(('contract ', 'interface ', 'library ')):
                    in_contract = True
                    current_contract = [line]
                    brace_count = line.count('{') - line.count('}')
            else:
                current_contract.append(line)
                brace_count += line.count('{') - line.count('}')
            if in_contract and brace_count == 0:
                contracts.append('\n'.join(current_contract).strip())
                current_contract = []
                in_contract = False
        return contracts

    def get_contract_name(contract_code):
        lines = contract_code.split('\n')
        first_line = lines[0].strip() if lines else ''
        parts = first_line.split()
        if len(parts) < 2 or parts[0] not in ['contract', 'interface', 'library']:
            return None
        name_part = parts[1]
        name = name_part.split('{')[0].split('is')[0].strip()
        return name

    # Process each contract to extract pragma and contracts
    for contract in contracts:
        code = contract.strip()
        lines = code.split('\n')
        pragma_lines = []
        in_pragma = False

        # Extract pragma lines
        for line in lines:
            stripped = line.strip()
            if stripped.startswith('pragma'):
                pragma_lines.append(line)
                in_pragma = True
            elif in_pragma:
                break  # Stop after first non-pragma line following pragma

        # Add pragma to set
        pragma_str = '\n'.join(pragma_lines).strip()
        if pragma_str:
            pragma_set.add(pragma_str)

        # Extract rest of the code and process contracts
        rest_code = '\n'.join(lines[len(pragma_lines):]).strip()
        if rest_code:
            contracts_in_rest = extract_contracts(rest_code)
            extracted_contracts.extend(contracts_in_rest)

    # Deduplicate contracts by name, preserving order
    seen = set()
    deduped_contracts = []
    for contract in extracted_contracts:
        name = get_contract_name(contract)
        if name and name not in seen:
            seen.add(name)
            deduped_contracts.append(contract)

    # Build merged code
    merged_code = []
    # Handle pragma directives
    if len(pragma_set) == 1:
        merged_code.append(next(iter(pragma_set)))
    else:
        merged_code.extend(sorted(pragma_set))  # Sort for consistency if multiple pragmas

    # Add deduplicated contracts
    merged_code.extend(deduped_contracts)

    return '\n\n'.join(merged_code).strip()

In [ ]:
sol_dict = {}
for info in data:
    sol_dict[info['idx']] = (info['address'], info['contract'])

In [ ]:
def process_data(filepath):
    """Processes a data file and returns lists of contracts classified as 0 and 1."""
    with open(filepath, 'r') as f:
        for line in f:
            try:
                index, label = line.strip().split()
                label = int(label)
                address, contract = sol_dict[index]
                sol_dict[index] = (address, contract, label)
            except ValueError:
                print(f"Skipping invalid line: {line.strip()}")

# Example usage:
process_data('train.txt')
process_data('valid.txt')
process_data('test.txt')

In [ ]:
contract_groups = []
current_address = ""
current_group = []
for idx in range(len(sol_dict)):
    address, contract, label = sol_dict[str(idx)]
    if address != current_address:
        if current_address != "":
            contract_groups.append((current_address, current_group))
        current_address = address
        current_group = []
    current_group.append((contract, label))

In [ ]:
print(len(contract_groups))

46056


In [ ]:
peculiar_files = {}
unpeculiar_files = {}

for adr, gr in contract_groups:
    cp = 0
    contracts = [k for k, v in gr]
    for contract, label in gr:
        if label == 1:
            cp = 1
            break

    merged_sc = merge_contracts(contracts).replace(" _\n", " _;\n")
    merged_sc = remove_comments(merged_sc)
    if cp == 1:
        peculiar_files[merged_sc] = adr
    else:
        unpeculiar_files[merged_sc] = adr

In [ ]:
print(len(peculiar_files))
print(len(unpeculiar_files))

1104
44442


In [ ]:
peculiar_reentrancy_count = []
for file_content in peculiar_files.keys():
    if file_content:  # Check if file_content is not empty
        old_matches = re.findall(r"call\.value", file_content)
        new_matches = re.findall(r"call{value", file_content)
        ok = 0
        if (len(old_matches) == 1 and not new_matches) or \
           (len(new_matches) == 1 and not old_matches):
            ok = 1
        if ok == 1:
            lines = file_content.splitlines(keepends=True)
            for i, line in enumerate(lines):
                if "call.value" in line or "call{value" in line:
                  try:
                      peculiar_reentrancy_count.append(find_function(lines, i))
                  except:
                      pass

print("Number of reentrancy files with one 'call.value':", len(set(peculiar_reentrancy_count)))

Number of reentrancy files with one 'call.value': 338


## Messi-Q / DeeSCVHunter

In [ ]:
reentrancy_files = os.listdir("/content/DeeSCVHunter/preprocessing/data/reentrancy/solidity_contract")
timedep_files =  os.listdir("/content/DeeSCVHunter/preprocessing/data/timestamp/solidity_contract")

deescv_files = {
    "reentrancy": {},
    "timestamp": {}
}

for vul in deescv_files.keys():
    files = os.listdir(f"/content/DeeSCVHunter/preprocessing/data/{vul}/solidity_contract")
    for file in files:
        with open(f"/content/DeeSCVHunter/preprocessing/data/{vul}/solidity_contract/{file}", "r") as f:
            code = f.read()
        file_name = file.split(".")[0]
        code = remove_comments(code)
        deescv_files[vul][code] = file_name

In [ ]:
print(len(deescv_files["reentrancy"]))
print(len(deescv_files["timestamp"]))

185
150


In [ ]:
deescv_reentrancy_count = []
for file_content in deescv_files["reentrancy"].keys():
    if file_content:  # Check if file_content is not empty
        old_matches = re.findall(r"call\.value", file_content)
        new_matches = re.findall(r"call{ value", file_content)
        ok = 0
        if (len(old_matches) == 1 and not new_matches) or \
           (len(new_matches) == 1 and not old_matches):
            ok = 1
        if ok == 1:
            lines = file_content.splitlines(keepends=True)
            for i, line in enumerate(lines):
                if "call.value" in line:
                  try:
                      deescv_reentrancy_count.append(remove_comments(find_function(lines, i)))
                  except:
                      pass

deescv_timedep_count = []
for file_content in deescv_files["timestamp"].keys():
    if file_content:  # Check if file_content is not empty
        now_matches = re.findall(r"now", file_content)
        timestamp_matches = re.findall(r"block\.timestamp", file_content)
        ok = 0
        if (len(now_matches) == 1 and not timestamp_matches) or \
           (len(timestamp_matches) == 1 and not now_matches):
            ok = 1
        if ok == 1:
            lines = file_content.splitlines(keepends=True)
            for i, line in enumerate(lines):
                if "now" in line or "block.timestamp" in line:
                  try:
                      deescv_timedep_count.append(remove_comments(find_function(lines, i)))
                  except:
                      pass

print("Number of reentrancy files with one 'call.value':", len(set(deescv_reentrancy_count)))
print("Number of time dependency files with one 'now' or 'block.timestamp':", len(set(deescv_timedep_count)))

Number of reentrancy files with one 'call.value': 156
Number of time dependency files with one 'now' or 'block.timestamp': 21


## SolAudit Dataset

In [ ]:
# Replace with your Kaggle API credentials
!mkdir -p ~/.kaggle
!echo '{"username":"QuangNguyen711","key":"96bcd39c738542c30dcaf6019e64d695"}' > ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

# Replace with the dataset URL from Kaggle
!kaggle datasets download -d 'jakeclark38a/soliaudit-va-dataset-sourcecode'

Dataset URL: https://www.kaggle.com/datasets/jakeclark38a/soliaudit-va-dataset-sourcecode
License(s): unknown
 97% 37.0M/38.1M [00:00<00:00, 91.0MB/s]
100% 38.1M/38.1M [00:00<00:00, 74.8MB/s]


In [ ]:
# Unzip the downloaded dataset
!mkdir /content/soliaudit_dataset
!unzip /content/soliaudit-va-dataset-sourcecode.zip -d /content/soliaudit_dataset

Archive:  /content/soliaudit-va-dataset-sourcecode.zip
  inflating: /content/soliaudit_dataset/SoliAudit-VA-Dataset-SourceCode.csv  


In [ ]:
import pandas as pd

df = pd.read_csv("/content/soliaudit_dataset/SoliAudit-VA-Dataset-SourceCode.csv")
df

,Addr,Underflow,Overflow,CallDepth,TOD,TimeDep,Reentracy,AssertFail,CheckEffects,InlineAssembly,BlockTimestamp,LowlevelCalls,SelfDestruct,source_code
0,0x0000000000b3F879cb30FE243b4Dfee438691c04,1,1,1,0,0,0,0,1,1,0,1,0,pragma solidity ^0.4.10;\n\ncontract GasToken2...
1,0x000000002647e16d9bab9e46604d75591d289277,1,1,0,1,1,0,1,1,1,1,0,0,pragma solidity ^0.4.20;// blaze it\n\ninterfa...
2,0x000000002bb43c83ece652d161ad0fa862129a2c,1,1,0,1,1,0,1,1,1,1,0,0,pragma solidity ^0.4.20;// blaze it\n\ninterfa...
3,0x000000005fbe2cc9b1b684ec445caf176042348e,1,1,0,1,1,0,1,1,1,1,0,0,pragma solidity ^0.4.20;// blaze it\n\ninterfa...
4,0x0006157838d5a6b33ab66588a6a693a57c869999,0,0,0,0,0,0,0,0,0,0,0,0,pragma solidity ^0.4.11;\n\ncontract IconomiBl...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17974,0xffed1cb392e36bdf5004450ab76fab23a5337c05,1,1,0,0,0,0,0,0,0,0,1,0,pragma solidity ^0.4.4;\n\ncontract Token {\n\...
17975,0xfff18893bc6c430741b3c5ad483391a5e21220bb,0,1,0,0,0,0,1,0,0,0,0,0,pragma solidity ^0.4.13;\n\n\n/**\n * @title S...
17976,0xfff3d3b591e792eb3d937327b4b786db37ba2087,1,1,0,0,0,0,0,0,0,0,0,0,pragma solidity ^0.4.13;\ncontract owned {\n ...
17977,0xfffce2dc587badbd10b4fe17f0f5f293458f6793,0,1,0,0,0,0,1,1,0,1,0,0,pragma solidity ^0.4.18;\n\ncontract Admin {\n...


In [ ]:
soliaudit_reentrancy_files = {}
soliaudit_timedep_files = {}
soliaudit_non_timedep_files = {}

for index, row in df.iterrows():
    sc = remove_comments(row['source_code'])
    if row['Reentracy'] == 1:
        # soliaudit_reentrancy_files.append((row['source_code'], row['source_code']))
        soliaudit_reentrancy_files[sc] = row['Addr']
    if row['BlockTimestamp'] == 1:
        # soliaudit_timedep_files.append((row['source_code'], row['source_code']))
        soliaudit_timedep_files[sc] = row['Addr']
    if row['TimeDep'] == 1:
        # soliaudit_timedep_files.append((row['source_code'], row['source_code']))
        soliaudit_timedep_files[sc] = row['Addr']
    if row['TimeDep'] == 0 and row['BlockTimestamp'] == 0:
        soliaudit_non_timedep_files[sc] = row['Addr']


print("Reentrancy files:", len(soliaudit_reentrancy_files))
print("Time Dependency files:", len(soliaudit_timedep_files))
print("Non Time Dependency files:", len(soliaudit_non_timedep_files))

Reentrancy files: 439
Time Dependency files: 4768
Non Time Dependency files: 11172


In [ ]:
from types import new_class
# prompt: In reentrancy_files check how many files have only one "call.value" and check in timedep_files check how many files have only one "now" or one "block.timestamp"

import re

soliaudit_timedep_count = []
for file_content, adr in soliaudit_timedep_files.items():
    if file_content:  # Check if file_content is not empty
        now_matches = re.findall(r"now", file_content)
        timestamp_matches = re.findall(r"block\.timestamp", file_content)
        ok = 0
        if (len(now_matches) == 1 and not timestamp_matches) or \
           (len(timestamp_matches) == 1 and not now_matches):
            ok = 1
        if ok == 1:
            lines = file_content.splitlines(keepends=True)
            for i, line in enumerate(lines):
                if "now" in line or "block.timestamp" in line:
                  try:
                      soliaudit_timedep_count.append(remove_comments(find_function(lines, i)))
                  except:
                      pass

print("Number of time dependency files with one 'now' or 'block.timestamp':", len(set(soliaudit_timedep_count)))

Number of time dependency files with one 'now' or 'block.timestamp': 454


## Cleaning

In [ ]:
total_reentrancy_funcs = list(set(smartbugs_functions["reentrancy"]["vulnerable"])) + list(set(peculiar_reentrancy_count)) + list(set(deescv_reentrancy_count))
undup_reentrancy_funcs = list(set(total_reentrancy_funcs))
print(len(total_reentrancy_funcs))
print(len(undup_reentrancy_funcs))
total_timedep_funcs = list(set(smartbugs_functions["time_manipulation"]["vulnerable"])) + list(set(deescv_timedep_count)) + list(set(soliaudit_timedep_count))
undup_timedep_funcs = list(set(total_timedep_funcs))
print(len(total_timedep_funcs))
print(len(undup_timedep_funcs))

total_reentrancy_code = {}
for sc, adr in smartbugs_files["reentrancy"].items():
    sc = remove_comments(sc)
    if sc not in total_reentrancy_code and "call.value" in sc:
        total_reentrancy_code[sc] = adr
for sc, adr in peculiar_files.items():
    sc = remove_comments(sc)
    if sc not in total_reentrancy_code and "call.value" in sc:
        total_reentrancy_code[sc] = adr
for sc, adr in deescv_files["reentrancy"].items():
    sc = remove_comments(sc)
    if sc not in total_reentrancy_code and "call.value" in sc:
        total_reentrancy_code[sc] = adr

total_timedep_code = {}
for sc, adr in smartbugs_files["time_manipulation"].items():
    sc = remove_comments(sc)
    if sc not in total_timedep_code:
        total_timedep_code[sc] = adr
for sc, adr in deescv_files["timestamp"].items():
    sc = remove_comments(sc)
    if sc not in total_timedep_code:
        total_timedep_code[sc] = adr
for sc, adr in soliaudit_timedep_files.items():
    sc = remove_comments(sc)
    if sc not in total_timedep_code:
        total_timedep_code[sc] = adr

508
508
481
476


In [ ]:
print(len(total_reentrancy_code))
print(len(total_timedep_code))

1262
4923


### Get Re-entrancy Dataset (Source code tập test thì không cần chứa function tập test đâu nhé sửa lại đi)

In [ ]:
import random

best_seed = -1
best_preserve = -1

for seed in range(100):
    random.seed(seed)
    # Assuming undup_reentrancy_funcs is defined in the previous code
    # Split undup_reentrancy_funcs into train and test sets
    shuffled_funcs = undup_reentrancy_funcs.copy()
    random.shuffle(shuffled_funcs)
    split_index = int(0.8 * len(shuffled_funcs))
    reen_train_vul_funcs = shuffled_funcs[:split_index]
    reen_test_vul_funcs = shuffled_funcs[split_index:]

    # print("Number of train functions:", len(reen_train_vul_funcs))
    # print("Number of test functions:", len(reen_test_vul_funcs))

    # Count occurrences of train functions in total_reentrancy_code
    reen_train_vul_count = []
    for code, adr in total_reentrancy_code.items():
        cp = 0
        for func in reen_train_vul_funcs:
            if func in code:
                cp = 1
                break
        for func2 in reen_test_vul_funcs:
            if func2 in code:
                cp = 0
                break
        if cp == 1:
            reen_train_vul_count.append((adr, code))

    # Count occurrences of test functions in total_reentrancy_code
    reen_test_vul_count = []
    for code, adr in total_reentrancy_code.items():
        cp = 0
        for func in reen_test_vul_funcs:
            if func in code and (adr, code) not in reen_train_vul_count:
                cp = 1
                break
        for func2 in reen_train_vul_funcs:
            if func2 in code:
                cp = 0
                break
        if cp == 1:
            reen_test_vul_count.append((adr, code))

    if len(reen_train_vul_count) / len(reen_test_vul_count) >= 3.8 and len(reen_train_vul_count) / len(reen_test_vul_count) <= 4.2:
        if len(reen_train_vul_count) + len(reen_test_vul_count) > best_preserve:
            best_preserve = len(reen_train_vul_count) + len(reen_test_vul_count)
            best_seed = seed
        print(f"Seed {seed} is compatible ~{round(len(reen_train_vul_count) / len(reen_test_vul_count), 2)} with able to retain {len(reen_train_vul_count) + len(reen_test_vul_count)} out of {len(total_reentrancy_code)}")
        # break

Seed 6 is compatible ~3.98 with able to retain 1015 out of 1262
Seed 28 is compatible ~4.1 with able to retain 1040 out of 1262
Seed 30 is compatible ~3.81 with able to retain 987 out of 1262
Seed 42 is compatible ~4.06 with able to retain 1047 out of 1262
Seed 48 is compatible ~3.96 with able to retain 1017 out of 1262
Seed 55 is compatible ~4.09 with able to retain 1028 out of 1262
Seed 61 is compatible ~3.89 with able to retain 973 out of 1262
Seed 70 is compatible ~4.1 with able to retain 994 out of 1262
Seed 71 is compatible ~3.97 with able to retain 1013 out of 1262
Seed 76 is compatible ~3.82 with able to retain 993 out of 1262
Seed 86 is compatible ~3.83 with able to retain 1025 out of 1262
Seed 93 is compatible ~3.82 with able to retain 1013 out of 1262
Seed 96 is compatible ~4.07 with able to retain 1004 out of 1262


In [ ]:
def get_filtered_data(data_dict, data_num):
    filtered_data = []
    i = 0
    step = 0
    key_list = list(data_dict.keys())
    while len(filtered_data) < data_num:
        if i > len(key_list) - 1:
            i = 0
            step += 1
        if step < len(data_dict[key_list[i]]):
            filtered_data.append(data_dict[key_list[i]][step])
        i += 1
    return filtered_data

In [ ]:
import random

random.seed(best_seed)
# Assuming undup_reentrancy_funcs is defined in the previous code
# Split undup_reentrancy_funcs into train and test sets -----------------------------------------------------------------------------------------------------------------------------
shuffled_funcs = undup_reentrancy_funcs.copy()
random.shuffle(shuffled_funcs)
split_index = int(0.8 * len(shuffled_funcs))
reen_train_vul_funcs = shuffled_funcs[:split_index]
reen_test_vul_funcs = shuffled_funcs[split_index:]

print("Number of train functions:", len(reen_train_vul_funcs))
print("Number of test functions:", len(reen_test_vul_funcs))

# Check occurrences of vul train functions in total_reentrancy_code -----------------------------------------------------------------------------------------------------------------
reen_train_vul_count = []
reen_train_vul_dict = {}

for code, adr in total_reentrancy_code.items():
    cp = 0
    for func in reen_train_vul_funcs:
        if func in code:
            cp = 1
            break
    for func2 in reen_test_vul_funcs:
        if func2 in code:
            cp = 0
            break
    if cp == 1:
        reen_train_vul_count.append((adr, code))
        if func not in reen_train_vul_dict:
            reen_train_vul_dict[func] = []
        reen_train_vul_dict[func].append((adr, code))

# Check occurrences of vul test functions in total_reentrancy_code -----------------------------------------------------------------------------------------------------------------
reen_test_vul_count = []
reen_test_vul_dict = {}

for code, adr in total_reentrancy_code.items():
    cp = 0
    for func in reen_test_vul_funcs:
        if func in code and (adr, code) not in reen_train_vul_count:
            cp = 1
            break
    for func2 in reen_train_vul_funcs:
        if func2 in code:
            cp = 0
            break
    if cp == 1:
        reen_test_vul_count.append((adr, code))
        if func not in reen_test_vul_dict:
            reen_test_vul_dict[func] = []
        reen_test_vul_dict[func].append((adr, code))

# Check number of reentrancy vul source codes that have more than 1 call.value -----------------------------------------------------------------------------------------------------
reen_vul_not_counted = []

for code, adr in total_reentrancy_code.items():
    if (adr, code) not in reen_train_vul_count and (adr, code) not in reen_test_vul_count:
        reen_vul_not_counted.append((adr, code))

random.shuffle(reen_vul_not_counted)
split_index = int(0.8 * len(reen_vul_not_counted))
reen_train_vul_not_counted = reen_vul_not_counted[:split_index]
reen_test_vul_not_counted = reen_vul_not_counted[split_index:]

# reen_train_vul_count = get_filtered_data(reen_train_vul_dict, )

print("Number of vulnerability source codes with 1 call.value containing train functions:", len(reen_train_vul_count))
print("Number of vulnerability source codes with 1 call.value containing test functions:", len(reen_test_vul_count))
print("Number of vulnerability source codes with more than 1 call.value containing train functions:", len(reen_train_vul_not_counted))
print("Number of vulnerability source codes with more than 1 call.value containing test functions:", len(reen_test_vul_not_counted))

# Get all source code that doesn't contain reentrancy in puculiar dataset --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

unpeculiar_syntax_files = []
unpeculiar_not_syntax_files = []
for sc, adr in unpeculiar_files.items():
    if "call.value" in sc:
        unpeculiar_syntax_files.append((adr, sc))
    else:
        unpeculiar_not_syntax_files.append((adr, sc))

# Get non-vulnerability functions still contain syntax call.value and functions that not contain syntax call.value ----------------------------------------------------------------------------
reen_non_vul_syntax_funcs = []
reen_non_vul_not_syntax_funcs = []

check_source_code_syntax_num = []

for adr, sc in unpeculiar_syntax_files:
    lines = sc.splitlines(keepends=True)
    cp = 0
    for i, line in enumerate(lines):
        if "call.value" in line:
            cp = 1
            try:
                reen_non_vul_syntax_funcs.append(find_function(lines, i))
            except:
                pass
        else:
            try:
                reen_non_vul_not_syntax_funcs.append(find_function(lines, i))
            except:
                pass
    if cp == 1:
        check_source_code_syntax_num.append(adr)

print("Number of non-vulnerability source codes containing syntax call.value:", len(check_source_code_syntax_num))

reen_non_vul_syntax_funcs = list(set(reen_non_vul_syntax_funcs))
reen_non_vul_not_syntax_funcs = list(set(reen_non_vul_not_syntax_funcs))

print("Number of non-vulnerability functions containing syntax call.value:", len(reen_non_vul_syntax_funcs))
print("Number of non-vulnerability functions not containing syntax call.value:", len(reen_non_vul_not_syntax_funcs))
# --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

Number of train functions: 406
Number of test functions: 102
Number of vulnerability source codes with 1 call.value containing train functions: 840
Number of vulnerability source codes with 1 call.value containing test functions: 207
Number of vulnerability source codes with more than 1 call.value containing train functions: 172
Number of vulnerability source codes with more than 1 call.value containing test functions: 43
Number of non-vulnerability source codes containing syntax call.value: 405
Number of non-vulnerability functions containing syntax call.value: 264
Number of non-vulnerability functions not containing syntax call.value: 7736


In [ ]:
best_seed = -1
best_preserve = -1

for seed in range(100):
    random.seed(seed)
    shuffled_funcs = reen_non_vul_syntax_funcs.copy()
    random.shuffle(shuffled_funcs)
    split_index = int(0.8 * len(shuffled_funcs))
    reen_train_non_vul_syntax_funcs = shuffled_funcs[:split_index]
    reen_test_non_vul_syntax_funcs = shuffled_funcs[split_index:]

    reen_sampled_not_syntax_funcs = random.sample(reen_non_vul_not_syntax_funcs, len(reen_train_vul_funcs + reen_test_vul_funcs) - len(reen_train_non_vul_syntax_funcs + reen_test_non_vul_syntax_funcs))
    random.shuffle(reen_sampled_not_syntax_funcs)
    split_index = int(0.8 * len(reen_sampled_not_syntax_funcs))
    reen_train_non_vul_not_syntax_funcs = reen_sampled_not_syntax_funcs[:split_index]
    reen_test_non_vul_not_syntax_funcs = reen_sampled_not_syntax_funcs[split_index:]
    reen_train_non_vul_syntax_count = []
    reen_test_non_vul_syntax_count = []

    # Count occurrences of train functions in total_reentrancy_code
    for adr, code in unpeculiar_syntax_files:
        cp = 0
        for func in reen_train_non_vul_syntax_funcs:
            if func in code:
                cp = 1
                break
        for func2 in reen_test_non_vul_syntax_funcs:
            if func2 in code:
                cp = 0
                break
        if cp == 1:
            reen_train_non_vul_syntax_count.append((adr, code))


    # Count occurrences of test functions in total_reentrancy_code
    for adr, code in unpeculiar_syntax_files:
        cp = 0
        for func in reen_test_non_vul_syntax_funcs:
            if func in code and (adr, code) not in reen_train_non_vul_syntax_count:
                cp = 1
                break
        for func2 in reen_train_non_vul_syntax_funcs:
            if func2 in code:
                cp = 0
                break
        if cp == 1:
            reen_test_non_vul_syntax_count.append((adr, code))


    # print("Number of non-vulnerability source codes contain call.value in train:", len(reen_train_non_vul_syntax_count))
    # print("Number of non-vulnerability source codes contain call.value in test:", len(reen_test_non_vul_syntax_count))

    if len(reen_train_non_vul_syntax_count) / len(reen_test_non_vul_syntax_count) >= 3.8 and len(reen_train_non_vul_syntax_count) / len(reen_test_non_vul_syntax_count) <= 4.2:
        if len(reen_train_non_vul_syntax_count) + len(reen_test_non_vul_syntax_count) > best_preserve:
            best_preserve = len(reen_train_non_vul_syntax_count) + len(reen_test_non_vul_syntax_count)
            best_seed = seed
        print(f"Seed {seed} is compatible ~{round(len(reen_train_non_vul_syntax_count) / len(reen_test_non_vul_syntax_count), 2)} with able to retain {len(reen_train_non_vul_syntax_count) + len(reen_test_non_vul_syntax_count)} out of {len(unpeculiar_syntax_files)}")
        # break

Seed 17 is compatible ~3.89 with able to retain 323 out of 405
Seed 35 is compatible ~4.07 with able to retain 360 out of 405
Seed 38 is compatible ~3.92 with able to retain 325 out of 405
Seed 52 is compatible ~3.82 with able to retain 352 out of 405
Seed 82 is compatible ~4.0 with able to retain 340 out of 405
Seed 88 is compatible ~3.83 with able to retain 348 out of 405
Seed 89 is compatible ~3.92 with able to retain 359 out of 405


In [ ]:
random.seed(best_seed)
shuffled_funcs = reen_non_vul_syntax_funcs.copy()
random.shuffle(shuffled_funcs)
split_index = int(0.8 * len(shuffled_funcs))
reen_train_non_vul_syntax_funcs = shuffled_funcs[:split_index]
reen_test_non_vul_syntax_funcs = shuffled_funcs[split_index:]

reen_sampled_not_syntax_funcs = random.sample(reen_non_vul_not_syntax_funcs, len(reen_train_vul_funcs + reen_test_vul_funcs) - len(reen_train_non_vul_syntax_funcs + reen_test_non_vul_syntax_funcs))
random.shuffle(reen_sampled_not_syntax_funcs)
split_index = int(0.8 * len(reen_sampled_not_syntax_funcs))
reen_train_non_vul_not_syntax_funcs = reen_sampled_not_syntax_funcs[:split_index]
reen_test_non_vul_not_syntax_funcs = reen_sampled_not_syntax_funcs[split_index:]
reen_train_non_vul_syntax_count, reen_test_non_vul_syntax_count = [], []
reen_train_non_vul_syntax_dict, reen_test_non_vul_syntax_dict = {}, {}

# Check occurrences of non-vul train functions in unpeculiar_syntax_files -----------------------------------------------------------------------------------------------------------------
for adr, code in unpeculiar_syntax_files:
    cp = 0
    for func in reen_train_non_vul_syntax_funcs:
        if func in code:
            cp = 1
            break
    for func2 in reen_test_non_vul_syntax_funcs:
        if func2 in code:
            cp = 0
            break
    if cp == 1:
        reen_train_non_vul_syntax_count.append((adr, code))
        if func not in reen_train_non_vul_syntax_dict:
            reen_train_non_vul_syntax_dict[func] = []
        reen_train_non_vul_syntax_dict[func].append((adr, code))

# Check occurrences of non-vul test functions in unpeculiar_syntax_files ------------------------------------------------------------------------------------------------------------------
for adr, code in unpeculiar_syntax_files:
    cp = 0
    for func in reen_test_non_vul_syntax_funcs:
        if func in code and (adr, code) not in reen_train_non_vul_syntax_count:
            cp = 1
            break
    for func2 in reen_train_non_vul_syntax_funcs:
        if func2 in code:
            cp = 0
            break
    if cp == 1:
        reen_test_non_vul_syntax_count.append((adr, code))
        if func not in reen_test_non_vul_syntax_dict:
            reen_test_non_vul_syntax_dict[func] = []
        reen_test_non_vul_syntax_dict[func].append((adr, code))

print("Number of non-vulnerability source codes contain call.value in train:", len(reen_train_non_vul_syntax_count))
print("Number of non-vulnerability source codes contain call.value in test:", len(reen_test_non_vul_syntax_count))

Number of non-vulnerability source codes contain call.value in train: 289
Number of non-vulnerability source codes contain call.value in test: 71


In [ ]:
print("Number of vulnerability source codes with 1 call.value containing train functions:", len(reen_train_vul_count))
print("Number of vulnerability source codes with 1 call.value containing test functions:", len(reen_test_vul_count))

print("Number of vulnerability source codes with more than 1 call.value containing train functions:", len(reen_train_vul_not_counted))
print("Number of vulnerability source codes with more than 1 call.value containing test functions:", len(reen_test_vul_not_counted))

print("Number of non-vulnerability source codes contain call.value in train:", len(reen_train_non_vul_syntax_count))
print("Number of non-vulnerability source codes contain call.value in test:", len(reen_test_non_vul_syntax_count))

Number of vulnerability source codes with 1 call.value containing train functions: 840
Number of vulnerability source codes with 1 call.value containing test functions: 207
Number of vulnerability source codes with more than 1 call.value containing train functions: 172
Number of vulnerability source codes with more than 1 call.value containing test functions: 43
Number of non-vulnerability source codes contain call.value in train: 289
Number of non-vulnerability source codes contain call.value in test: 71


In [ ]:
reen_train_vul_total = get_filtered_data(reen_train_vul_dict, int(len(reen_train_non_vul_syntax_count) * 2/3))
reen_train_vul_total += random.sample(reen_train_vul_not_counted, int(len(reen_train_non_vul_syntax_count) * 1/3))
reen_test_vul_total = get_filtered_data(reen_test_vul_dict, int(len(reen_test_non_vul_syntax_count) * 2/3)) + random.sample(reen_test_vul_not_counted, int(len(reen_test_non_vul_syntax_count) * 1/3))
reen_train_non_vul_total = reen_train_non_vul_syntax_count
reen_test_non_vul_total = reen_test_non_vul_syntax_count

print("Number of vulnerability source codes in train:", len(reen_train_vul_total))
print("Number of vulnerability source codes in test:", len(reen_test_vul_total))
print("Number of non-vulnerability source codes in train:", len(reen_train_non_vul_total))
print("Number of non-vulnerability source codes in test:", len(reen_test_non_vul_total))

Number of vulnerability source codes in train: 288
Number of vulnerability source codes in test: 70
Number of non-vulnerability source codes in train: 289
Number of non-vulnerability source codes in test: 71


In [ ]:
cnt = 0
for adr, src in reen_train_non_vul_total:
    if "call.value" in src:
        cnt += 1

print(cnt)

289


In [ ]:
cnt = 0
for adr, src in reen_train_non_vul_total:
    if "call.value" in remove_comments(src):
        cnt += 1

print(cnt)

289


In [ ]:
reen_train_non_vul_funcs = reen_train_non_vul_syntax_funcs + reen_train_non_vul_not_syntax_funcs
reen_test_non_vul_funcs = reen_test_non_vul_syntax_funcs + reen_test_non_vul_not_syntax_funcs

print("Number of non-vulnerability functions in train:", len(reen_train_non_vul_funcs))
print("Number of non-vulnerability functions in test:", len(reen_test_non_vul_funcs))

Number of non-vulnerability functions in train: 406
Number of non-vulnerability functions in test: 102


### Get Timestamp Dependency Dataset

In [ ]:
import random

best_seed = -1
best_preserve = -1

for seed in range(100):
    random.seed(seed)
    # Assuming undup_timedep_funcs is defined in the previous code
    # Split undup_timedep_funcs into train and test sets -----------------------------------------------------------------------------------------------------------------
    shuffled_funcs = undup_timedep_funcs.copy()
    random.shuffle(shuffled_funcs)
    split_index = int(0.8 * len(shuffled_funcs))
    time_train_vul_funcs = shuffled_funcs[:split_index]
    time_test_vul_funcs = shuffled_funcs[split_index:]

    print("Number of train functions:", len(time_train_vul_funcs))
    print("Number of test functions:", len(time_test_vul_funcs))

    # Count occurrences of train functions in total_timedep_code -----------------------------------------------------------------------------------------------------------------
    time_train_vul_count = []
    for code, adr in total_timedep_code.items():
        cp = 0
        for func in time_train_vul_funcs:
            if func in code:
                cp = 1
                break
        for func2 in time_test_vul_funcs:
            if func2 in code:
                cp = 0
                break
        if cp == 1:
            time_train_vul_count.append((adr, code))

    # Count occurrences of test functions in total_timedep_code -----------------------------------------------------------------------------------------------------------------
    time_test_vul_count = []
    for code, adr in total_timedep_code.items():
        cp = 0
        for func in time_test_vul_funcs:
            if func in code and (adr, code) not in time_train_vul_count:
                cp = 1
                break
        for func2 in time_train_vul_funcs:
            if func2 in code:
                cp = 0
                break
        if cp == 1:
            time_test_vul_count.append((adr, code))

    if len(time_train_vul_count) / len(time_test_vul_count) >= 3.6 and len(time_train_vul_count) / len(time_test_vul_count) <= 4.2:
        if len(reen_train_vul_count) + len(reen_test_vul_count) > best_preserve:
            best_preserve = len(reen_train_vul_count) + len(reen_test_vul_count)
            best_seed = seed
        print(f"Seed {seed} is compatible ~{round(len(time_train_vul_count) / len(time_test_vul_count), 2)} with able to retain {len(time_train_vul_count) + len(time_test_vul_count)} out of {len(total_timedep_code)}")
        # break

Number of train functions: 380
Number of test functions: 96
Seed 0 is compatible ~4.03 with able to retain 2185 out of 4923
Number of train functions: 380
Number of test functions: 96
Number of train functions: 380
Number of test functions: 96
Number of train functions: 380
Number of test functions: 96
Number of train functions: 380
Number of test functions: 96
Number of train functions: 380
Number of test functions: 96
Number of train functions: 380
Number of test functions: 96
Number of train functions: 380
Number of test functions: 96
Number of train functions: 380
Number of test functions: 96
Number of train functions: 380
Number of test functions: 96
Number of train functions: 380
Number of test functions: 96
Number of train functions: 380
Number of test functions: 96
Number of train functions: 380
Number of test functions: 96
Number of train functions: 380
Number of test functions: 96
Number of train functions: 380
Number of test functions: 96
Number of train functions: 380
Numbe

In [ ]:
from datetime import time
import random

random.seed(best_seed)
# Assuming undup_timedep_funcs is defined in the previous code
# Split undup_timedep_funcs into train and test sets -----------------------------------------------------------------------------------------------------------------
shuffled_funcs = undup_timedep_funcs.copy()
random.shuffle(shuffled_funcs)
split_index = int(0.8 * len(shuffled_funcs))
time_train_vul_funcs = shuffled_funcs[:split_index]
time_test_vul_funcs = shuffled_funcs[split_index:]

print("Number of train functions:", len(time_train_vul_funcs))
print("Number of test functions:", len(time_test_vul_funcs))

print(len(total_timedep_code))
# Count occurrences of vul train functions in total_timedep_code -----------------------------------------------------------------------------------------------------------------
time_train_vul_count = []
time_train_vul_dict = {}

for code, adr in total_timedep_code.items():
    cp = 0
    for func in time_train_vul_funcs:
        if func in code:
            cp = 1
            break
    for func2 in time_test_vul_funcs:
        if func2 in code:
            cp = 0
            break
    if cp == 1:
        time_train_vul_count.append((adr, code))
        if func not in time_train_vul_dict:
            time_train_vul_dict[func] = []
        time_train_vul_dict[func].append((adr, code))

# Count occurrences of vul test functions in total_timedep_code -----------------------------------------------------------------------------------------------------------------
time_test_vul_count = []
time_test_vul_dict = {}

for code, adr in total_timedep_code.items():
    cp = 0
    for func in time_test_vul_funcs:
        if func in code and (adr, code) not in time_train_vul_count:
            cp = 1
            break
    for func2 in time_train_vul_funcs:
        if func2 in code:
            cp = 0
            break
    if cp == 1:
        time_test_vul_count.append((adr, code))
        if func not in time_test_vul_dict:
            time_test_vul_dict[func] = []
        time_test_vul_dict[func].append((adr, code))

# Check number of timestamp vul source codes that have more than 1 block.timestamp or now -----------------------------------------------------------------------------------------------------
time_vul_not_counted = []

for code, adr in total_timedep_code.items():
    if (adr, code) not in time_train_vul_count and (adr, code) not in time_test_vul_count:
        time_vul_not_counted.append((adr, code))

random.shuffle(time_vul_not_counted)
split_index = int(0.8 * len(time_vul_not_counted))
time_train_vul_not_counted = time_vul_not_counted[:split_index]
time_test_vul_not_counted = time_vul_not_counted[split_index:]

print("Number of vulnerability source codes with 1 block.timestamp or 1 now containing train functions:", len(time_train_vul_count))
print("Number of vulnerability source codes with 1 block.timestamp or 1 now containing test functions:", len(time_test_vul_count))
print("Number of vulnerability source codes with more than 1 block.timestamp or now containing train functions:", len(time_train_vul_not_counted))
print("Number of vulnerability source codes with more than 1 block.timestamp or now containing test functions:", len(time_test_vul_not_counted))
# Get all source code that doesn't contain timestamp dependency in puculiar dataset --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

soliaudit_syntax_files = []
soliaudit_not_syntax_files = []
for sc, adr in soliaudit_non_timedep_files.items():
    if "block.timestamp" in sc or "now" in sc:
        soliaudit_syntax_files.append((adr, sc))
    else:
        soliaudit_not_syntax_files.append((adr, sc))

# Get non-vulnerability functions still contain syntax (block.timestamp or now) and functions that not contain syntax (block.timestamp or now) ----------------------------------------------------------------------------
time_non_vul_syntax_funcs = []
time_non_vul_not_syntax_funcs = []
check_source_code_syntax_num = []

for adr, sc in soliaudit_syntax_files:
    lines = sc.splitlines(keepends=True)
    cp = 0
    for i, line in enumerate(lines):
        if "block.timestamp" in line or "now" in line:
            cp = 1
            try:
                time_non_vul_syntax_funcs.append(find_function(lines, i))
            except:
                pass
        else:
            try:
                time_non_vul_not_syntax_funcs.append(find_function(lines, i))
            except:
                pass
    if cp == 1:
        check_source_code_syntax_num.append(adr)

print("Number of non-vulnerability source codes containing syntax block.timestamp or now:", len(check_source_code_syntax_num))

time_non_vul_syntax_funcs = list(set(time_non_vul_syntax_funcs))
time_non_vul_not_syntax_funcs = list(set(time_non_vul_not_syntax_funcs))

print("Number of non-vulnerability functions containing syntax block.timestamp or now:", len(time_non_vul_syntax_funcs))
print("Number of non-vulnerability functions not containing syntax block.timestamp or now:", len(time_non_vul_not_syntax_funcs))

Number of train functions: 380
Number of test functions: 96
4923
Number of vulnerability source codes with 1 block.timestamp or 1 now containing train functions: 1751
Number of vulnerability source codes with 1 block.timestamp or 1 now containing test functions: 434
Number of vulnerability source codes with more than 1 block.timestamp or now containing train functions: 2190
Number of vulnerability source codes with more than 1 block.timestamp or now containing test functions: 548
Number of non-vulnerability source codes containing syntax block.timestamp or now: 148
Number of non-vulnerability functions containing syntax block.timestamp or now: 101
Number of non-vulnerability functions not containing syntax block.timestamp or now: 1893


In [ ]:
best_seed = -1
best_preserve = -1

for seed in range(200):
    random.seed(seed)
    sampled_syntax_funcs = time_non_vul_syntax_funcs.copy()
    random.shuffle(sampled_syntax_funcs)
    split_index = int(0.8 * len(sampled_syntax_funcs))
    time_train_non_vul_syntax_funcs = sampled_syntax_funcs[:split_index]
    time_test_non_vul_syntax_funcs = sampled_syntax_funcs[split_index:]

    sampled_not_syntax_funcs = random.sample(time_non_vul_not_syntax_funcs, len(time_train_vul_funcs + time_test_vul_funcs) - len(time_train_non_vul_syntax_funcs + time_test_non_vul_syntax_funcs))
    random.shuffle(sampled_not_syntax_funcs)
    split_index = int(0.8 * len(sampled_not_syntax_funcs))
    train_non_vul_not_syntax_funcs = sampled_not_syntax_funcs[:split_index]
    test_non_vul_not_syntax_funcs = sampled_not_syntax_funcs[split_index:]
    time_train_non_vul_syntax_count = []
    time_test_non_vul_syntax_count = []

    # Count occurrences of train functions in total_reentrancy_code
    for adr, code in soliaudit_syntax_files:
        cp = 0
        for func in time_train_non_vul_syntax_funcs:
            if func in code:
                cp = 1
                break
        for func2 in time_test_non_vul_syntax_funcs:
            if func2 in code:
                cp = 0
                break
        if cp == 1:
            time_train_non_vul_syntax_count.append((adr, code))


    # Count occurrences of test functions in total_reentrancy_code
    for adr, code in soliaudit_syntax_files:
        cp = 0
        for func in time_test_non_vul_syntax_funcs:
            if func in code and (adr, code) not in time_train_non_vul_syntax_count:
                cp = 1
                break
        for func2 in time_train_non_vul_syntax_funcs:
            if func2 in code:
                cp = 0
                break
        if cp == 1:
            time_test_non_vul_syntax_count.append((adr, code))

    # print("Number of non-vulnerability source codes contain block.timestamp or now in train:", len(time_train_non_vul_syntax_count))
    # print("Number of non-vulnerability source codes contain block.timestamp or now in test:", len(time_test_non_vul_syntax_count))

    if len(time_train_non_vul_syntax_count) / len(time_test_non_vul_syntax_count) >= 3.9 and len(time_train_non_vul_syntax_count) / len(time_test_non_vul_syntax_count) <= 4.2:
        if len(time_train_non_vul_syntax_count) + len(time_test_non_vul_syntax_count) > best_preserve:
            best_preserve = len(time_train_non_vul_syntax_count) + len(time_test_non_vul_syntax_count)
            best_seed = seed
        print(f"Seed {seed} is compatible ~{round(len(time_train_non_vul_syntax_count) / len(time_test_non_vul_syntax_count), 2)} with able to retain {len(time_train_non_vul_syntax_count) + len(time_test_non_vul_syntax_count)} out of {len(soliaudit_syntax_files)}")
        # break

Seed 64 is compatible ~4.16 with able to retain 98 out of 148
Seed 73 is compatible ~3.93 with able to retain 74 out of 148
Seed 74 is compatible ~3.93 with able to retain 74 out of 148
Seed 87 is compatible ~4.2 with able to retain 78 out of 148
Seed 100 is compatible ~4.13 with able to retain 77 out of 148
Seed 104 is compatible ~3.9 with able to retain 98 out of 148
Seed 115 is compatible ~3.94 with able to retain 79 out of 148
Seed 116 is compatible ~3.93 with able to retain 74 out of 148
Seed 119 is compatible ~4.08 with able to retain 66 out of 148
Seed 155 is compatible ~3.92 with able to retain 64 out of 148
Seed 187 is compatible ~4.07 with able to retain 71 out of 148
Seed 189 is compatible ~4.06 with able to retain 81 out of 148


In [ ]:
random.seed(best_seed)
sampled_syntax_funcs = time_non_vul_syntax_funcs.copy()
random.shuffle(sampled_syntax_funcs)
split_index = int(0.8 * len(sampled_syntax_funcs))
time_train_non_vul_syntax_funcs = sampled_syntax_funcs[:split_index]
time_test_non_vul_syntax_funcs = sampled_syntax_funcs[split_index:]

sampled_not_syntax_funcs = random.sample(time_non_vul_not_syntax_funcs, len(time_train_vul_funcs + time_test_vul_funcs) - len(time_train_non_vul_syntax_funcs + time_test_non_vul_syntax_funcs))
random.shuffle(sampled_not_syntax_funcs)
split_index = int(0.8 * len(sampled_not_syntax_funcs))
time_train_non_vul_not_syntax_funcs = sampled_not_syntax_funcs[:split_index]
time_test_non_vul_not_syntax_funcs = sampled_not_syntax_funcs[split_index:]
time_train_non_vul_syntax_count, time_test_non_vul_syntax_count = [], []
time_train_non_vul_syntax_dict, time_test_non_vul_syntax_dict = {}, {}

# Check occurrences of non-vul train functions in soliaudit_syntax_files -----------------------------------------------------------------------------------------------------------------
for adr, code in soliaudit_syntax_files:
    cp = 0
    for func in time_train_non_vul_syntax_funcs:
        if func in code:
            cp = 1
            break
    for func2 in time_test_non_vul_syntax_funcs:
        if func2 in code:
            cp = 0
            break
    if cp == 1:
        time_train_non_vul_syntax_count.append((adr, code))
        if func not in time_train_non_vul_syntax_dict:
            time_train_non_vul_syntax_dict[func] = []
        time_train_non_vul_syntax_dict[func].append((adr, code))


# Check occurrences of non-vul test functions in soliaudit_syntax_files -----------------------------------------------------------------------------------------------------------------
for adr, code in soliaudit_syntax_files:
    cp = 0
    for func in time_test_non_vul_syntax_funcs:
        if func in code and (adr, code) not in time_train_non_vul_syntax_count:
            cp = 1
            break
    for func2 in time_train_non_vul_syntax_funcs:
        if func2 in code:
            cp = 0
            break
    if cp == 1:
        time_test_non_vul_syntax_count.append((adr, code))
        if func not in time_test_non_vul_syntax_dict:
            time_test_non_vul_syntax_dict[func] = []
        time_test_non_vul_syntax_dict[func].append((adr, code))

print("Number of non-vulnerability source codes contain block.timestamp or now in train:", len(time_train_non_vul_syntax_count))
print("Number of non-vulnerability source codes contain block.timestamp or now in test:", len(time_test_non_vul_syntax_count))

Number of non-vulnerability source codes contain block.timestamp or now in train: 79
Number of non-vulnerability source codes contain block.timestamp or now in test: 19


In [ ]:
print("Number of vulnerability source codes with 1 call.value containing train functions:", len(time_train_vul_count))
print("Number of vulnerability source codes with 1 call.value containing test functions:", len(time_test_vul_count))

print("Number of vulnerability source codes with more than 1 call.value containing train functions:", len(time_train_vul_not_counted))
print("Number of vulnerability source codes with more than 1 call.value containing test functions:", len(time_test_vul_not_counted))

print("Number of non-vulnerability source codes contain call.value in train:", len(time_train_non_vul_syntax_count))
print("Number of non-vulnerability source codes contain call.value in test:", len(time_test_non_vul_syntax_count))

Number of vulnerability source codes with 1 call.value containing train functions: 1751
Number of vulnerability source codes with 1 call.value containing test functions: 434
Number of vulnerability source codes with more than 1 call.value containing train functions: 2190
Number of vulnerability source codes with more than 1 call.value containing test functions: 548
Number of non-vulnerability source codes contain call.value in train: 79
Number of non-vulnerability source codes contain call.value in test: 19


In [ ]:
time_train_vul_total = get_filtered_data(time_train_vul_dict, int(len(time_train_non_vul_syntax_count) * 1/2)) + random.sample(time_train_vul_not_counted, int(len(time_train_non_vul_syntax_count) * 1/2))
time_test_vul_total = get_filtered_data(time_test_vul_dict, int(len(time_test_non_vul_syntax_count) * 1/2)) + random.sample(time_test_vul_not_counted, int(len(time_test_non_vul_syntax_count) * 1/2))
time_train_non_vul_total = time_train_non_vul_syntax_count
time_test_non_vul_total = time_test_non_vul_syntax_count

print("Number of vulnerability source codes in train:", len(time_train_vul_total))
print("Number of vulnerability source codes in test:", len(time_test_vul_total))
print("Number of non-vulnerability source codes in train:", len(time_train_non_vul_total))
print("Number of non-vulnerability source codes in test:", len(time_test_non_vul_total))

Number of vulnerability source codes in train: 78
Number of vulnerability source codes in test: 18
Number of non-vulnerability source codes in train: 79
Number of non-vulnerability source codes in test: 19


In [ ]:
time_train_non_vul_funcs = time_train_non_vul_syntax_funcs + time_train_non_vul_not_syntax_funcs
time_test_non_vul_funcs = time_test_non_vul_syntax_funcs + time_test_non_vul_not_syntax_funcs

print("Number of non-vulnerability functions in train:", len(time_train_non_vul_funcs))
print("Number of non-vulnerability functions in test:", len(time_test_non_vul_funcs))

Number of non-vulnerability functions in train: 380
Number of non-vulnerability functions in test: 96


In [ ]:
cnt = 0
for adr, src in time_train_non_vul_total:
    if "block.timestamp" in src or "now" in src:
        cnt += 1

print(cnt)

79


In [ ]:
cnt = 0
for adr, src in time_train_non_vul_total:
    src = remove_comments(src)
    if "block.timestamp" in src or "now" in src:
        cnt += 1

print(cnt)

79


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os

src = "/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/NonVulnerable"

cnt = 0
for file in os.listdir(src):
    with open(src + "/" + file, "r") as f:
        data = f.read()
    if "block.timestamp" in data or "now" in data:
        cnt =+ 1

print(cnt)

cnt = 0
for file in os.listdir(src):
    with open(src + "/" + file, "r") as f:
        data = remove_comments(f.read())
    if "block.timestamp" in data or "now" in data:
        cnt =+ 1

print(cnt)

1


In [ ]:
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/NonVulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/Vulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/ReentrancyDataset")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/NonVulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/Vulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/TimestampDependencyDataset")))

281
285
281
285
2
312
313
312
313
2


In [ ]:
for file in os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable"):
    os.remove(os.path.join("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable", file))

for file in os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/NonVulnerable"):
    os.remove(os.path.join("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/NonVulnerable", file))

for file in os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/Vulnerable"):
    os.remove(os.path.join("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/Vulnerable", file))

for file in os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable"):
    os.remove(os.path.join("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable", file))

for file in os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/ReentrancyDataset"):
    os.remove(os.path.join("/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/ReentrancyDataset", file))

In [ ]:
for file in os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable"):
    os.remove(os.path.join("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable", file))

for file in os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/NonVulnerable"):
    os.remove(os.path.join("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/NonVulnerable", file))

for file in os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/Vulnerable"):
    os.remove(os.path.join("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/Vulnerable", file))

for file in os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable"):
    os.remove(os.path.join("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable", file))

for file in os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/TimestampDependencyDataset"):
    os.remove(os.path.join("/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/TimestampDependencyDataset", file))

In [ ]:
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/NonVulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/Vulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/ReentrancyDataset")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/NonVulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/Vulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/TimestampDependencyDataset")))

0
0
0
0
0
0
0
0
0
0


In [ ]:
# os.mkdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset")
# os.mkdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset")
# os.mkdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset")

# os.mkdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train")
# os.mkdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test")
# os.mkdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable")
# os.mkdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/NonVulnerable")
# os.mkdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/Vulnerable")
# os.mkdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable")

# os.mkdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train")
# os.mkdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test")
# os.mkdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable")
# os.mkdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/NonVulnerable")
# os.mkdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/Vulnerable")
# os.mkdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable")

# os.mkdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset")
# os.mkdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/ReentrancyDataset")
# os.mkdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/TimestampDependencyDataset")

In [ ]:
cnt = 0
for adr, sc in reen_train_vul_total:
    if "pragma solidity" not in sc:
        sc = "pragma solidity ^0.4.24;\n" + sc
        cnt += 1
    sc = clean_code_formatting(remove_comments(sc))
    with open(f"/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable/{adr}.sol", "w") as f:
        f.write(sc)
print(cnt)

cnt = 0
for adr, sc in reen_train_non_vul_total:
    if "pragma solidity" not in sc:
        sc = "pragma solidity ^0.4.24;\n" + sc
        cnt += 1
    sc = clean_code_formatting(remove_comments(sc))
    with open(f"/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/NonVulnerable/{adr}.sol", "w") as f:
        f.write(sc)
print(cnt)

cnt = 0
for adr, sc in time_train_vul_total:
    if "pragma solidity" not in sc:
        sc = "pragma solidity ^0.4.24;\n" + sc
        cnt += 1
    sc = clean_code_formatting(remove_comments(sc))
    with open(f"/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable/{adr}.sol", "w") as f:
        f.write(sc)
print(cnt)

cnt = 0
for adr, sc in time_train_non_vul_total:
    if "pragma solidity" not in sc:
        sc = "pragma solidity ^0.4.24;\n" + sc
        cnt += 1
    sc = clean_code_formatting(remove_comments(sc))
    with open(f"/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/NonVulnerable/{adr}.sol", "w") as f:
        f.write(sc)
print(cnt)

18
18
16
53


In [ ]:
cnt = 0
for adr, sc in reen_test_vul_total:
    if "pragma solidity" not in sc:
        sc = "pragma solidity ^0.4.24;\n" + sc
        cnt += 1
    sc = clean_code_formatting(remove_comments(sc))
    with open(f"/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/Vulnerable/{adr}.sol", "w") as f:
        f.write(sc)
print(cnt)

cnt = 0
for adr, sc in reen_test_non_vul_total:
    if "pragma solidity" not in sc:
        sc = "pragma solidity ^0.4.24;\n" + sc
        cnt += 1
    sc = clean_code_formatting(remove_comments(sc))
    with open(f"/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable/{adr}.sol", "w") as f:
        f.write(sc)
print(cnt)

cnt = 0
for adr, sc in time_test_vul_total:
    if "pragma solidity" not in sc:
        sc = "pragma solidity ^0.4.24;\n" + sc
        cnt += 1
    sc = clean_code_formatting(remove_comments(sc))
    with open(f"/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/Vulnerable/{adr}.sol", "w") as f:
        f.write(sc)
print(cnt)

cnt = 0
for adr, sc in time_test_non_vul_total:
    if "pragma solidity" not in sc:
        sc = "pragma solidity ^0.4.24;\n" + sc
        cnt += 1
    sc = clean_code_formatting(remove_comments(sc))
    with open(f"/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable/{adr}.sol", "w") as f:
        f.write(sc)
print(cnt)

3
6
4
9


In [ ]:
import pandas as pd

time_train_vul_funcs = [clean_code_formatting(remove_comments(func)) for func in time_train_vul_funcs]
time_train_non_vul_funcs = [clean_code_formatting(remove_comments(func)) for func in time_train_non_vul_funcs]
time_test_vul_funcs = [clean_code_formatting(remove_comments(func)) for func in time_test_vul_funcs]
time_test_non_vul_funcs = [clean_code_formatting(remove_comments(func)) for func in time_test_non_vul_funcs]

# Create dataframes for training set
train_vul_df = pd.DataFrame({'function': time_train_vul_funcs, 'label': 1})
train_non_vul_df = pd.DataFrame({'function': time_train_non_vul_funcs, 'label': 0})
train_df = pd.concat([train_vul_df, train_non_vul_df], ignore_index=True)
train_df = train_df.sample(frac=1).reset_index(drop=True)

# Create dataframes for test set
test_vul_df = pd.DataFrame({'function': time_test_vul_funcs, 'label': 1})
test_non_vul_df = pd.DataFrame({'function': time_test_non_vul_funcs, 'label': 0})
test_df = pd.concat([test_vul_df, test_non_vul_df], ignore_index=True)
test_df = test_df.sample(frac=1).reset_index(drop=True)

with open("/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/TimestampDependencyDataset/train.csv", "w") as f:
    train_df.to_csv(f, index=False)

with open("/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/TimestampDependencyDataset/test.csv", "w") as f:
    test_df.to_csv(f, index=False)

In [ ]:
reen_train_vul_funcs = [clean_code_formatting(remove_comments(func)) for func in reen_train_vul_funcs]
reen_train_non_vul_funcs = [clean_code_formatting(remove_comments(func)) for func in reen_train_non_vul_funcs]
reen_test_vul_funcs = [clean_code_formatting(remove_comments(func)) for func in reen_test_vul_funcs]
reen_test_non_vul_funcs = [clean_code_formatting(remove_comments(func)) for func in reen_test_non_vul_funcs]

# Create dataframes for training set
train_vul_df = pd.DataFrame({'function': reen_train_vul_funcs, 'label': 1})
train_non_vul_df = pd.DataFrame({'function': reen_train_non_vul_funcs, 'label': 0})
train_df = pd.concat([train_vul_df, train_non_vul_df], ignore_index=True)
train_df = train_df.sample(frac=1).reset_index(drop=True)

# Create dataframes for test set
test_vul_df = pd.DataFrame({'function': reen_test_vul_funcs, 'label': 1})
test_non_vul_df = pd.DataFrame({'function': reen_test_non_vul_funcs, 'label': 0})
test_df = pd.concat([test_vul_df, test_non_vul_df], ignore_index=True)
test_df = test_df.sample(frac=1).reset_index(drop=True)

with open("/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/ReentrancyDataset/train.csv", "w") as f:
    train_df.to_csv(f, index=False)

with open("/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/ReentrancyDataset/test.csv", "w") as f:
    test_df.to_csv(f, index=False)

In [ ]:
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/NonVulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/Vulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/ReentrancyDataset")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/NonVulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/Vulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable")))
print(len(os.listdir("/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/TimestampDependencyDataset")))

297
298
73
74
2
294
295
72
73
2


In [ ]:
%%capture
!pip install datasets

In [ ]:
from huggingface_hub import HfApi, Repository
import os
from google.colab import userdata
from datasets import Dataset, DatasetDict
import pandas as pd

# Load your CSV files
train_df = pd.read_csv('/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/ReentrancyDataset/train.csv')
test_df = pd.read_csv('/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/ReentrancyDataset/test.csv')

# Convert pandas DataFrame to Hugging Face Dataset
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Create a DatasetDict for train and test
dataset = DatasetDict({
    'train': train_dataset,
    'test': test_dataset
})

repo_name = "Quangnguyen711/Qualified_Syntax_Reentrancy_Dataset"  # Replace with your desired repo ID
dataset.push_to_hub(repo_name)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

README.md:   0%|          | 0.00/403 [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/Quangnguyen711/Qualified_Syntax_Reentrancy_Dataset/commit/28387253c5eab2bcdc611ab37235bc4a243d52a1', commit_message='Upload dataset', commit_description='', oid='28387253c5eab2bcdc611ab37235bc4a243d52a1', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Quangnguyen711/Qualified_Syntax_Reentrancy_Dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Quangnguyen711/Qualified_Syntax_Reentrancy_Dataset'), pr_revision=None, pr_num=None)

In [ ]:
# Load your CSV files
train_df = pd.read_csv('/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/TimestampDependencyDataset/train.csv')
test_df = pd.read_csv('/content/drive/MyDrive/SmartContractVulnerabilityDetection/FunctionSyntaxDataset/TimestampDependencyDataset/test.csv')

# Convert pandas DataFrame to Hugging Face Dataset
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Create a DatasetDict for train and test
dataset = DatasetDict({
    'train': train_dataset,
    'test': test_dataset
})

repo_name = "Quangnguyen711/Qualified_Syntax_TimestampDependency_Dataset"  # Replace with your desired repo ID
dataset.push_to_hub(repo_name)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

README.md:   0%|          | 0.00/403 [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/Quangnguyen711/Qualified_Syntax_TimestampDependency_Dataset/commit/2e1568f880489bd9de7914e87a3dc10d1a34a4d0', commit_message='Upload dataset', commit_description='', oid='2e1568f880489bd9de7914e87a3dc10d1a34a4d0', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Quangnguyen711/Qualified_Syntax_TimestampDependency_Dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Quangnguyen711/Qualified_Syntax_TimestampDependency_Dataset'), pr_revision=None, pr_num=None)

In [ ]:
!zip -r /content/SmartContractVulnerabilityDetection.zip /content/drive/MyDrive/SmartContractVulnerabilityDetection

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
  adding: content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeDataset/TimestampDependencyDataset/Train/Vulnerable/0xc49501f07c9a587adac93e09ecae5f701098ec2a.sol (deflated 75%)
  adding: content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeDataset/TimestampDependencyDataset/Train/Vulnerable/0xc49e03bdd6809fd168565b26d27d5cf72f9e9525.sol (deflated 77%)
  adding: content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeDataset/TimestampDependencyDataset/Train/Vulnerable/0xc4aad17558fa95c8937d0856b2dad74c1a7a095f.sol (deflated 77%)
  adding: content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeDataset/TimestampDependencyDataset/Train/Vulnerable/0xc4d89062f53f20e9c50470e1cd0714ff161a4b3f.sol (deflated 78%)
  adding: content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeDataset/TimestampDependencyDataset/Train/Vulnerable/0xc4e8cb21436e79ba9ed0abeaec010a1982125b80.so

In [ ]:
while True:
  pass

KeyboardInterrupt: 

## SmartBugs Wild (LLM Pipeline Filter Needed)

In [ ]:
# prompt: Read all of the files in the dir /content/smartbugs-wild/contracts that ends with .sol, get 2 list of sol files, 1 contains call.value in the source code, 1 contains block.timestamp in the source code

import os

def get_sol_files_with_keywords(directory):
    """
    Reads all .sol files in the given directory and its subdirectories,
    and returns two lists: one containing files with "call.value",
    and another with files containing "block.timestamp".
    """
    files_with_call_value = []
    files_with_block_timestamp = []

    for root, _, files in os.walk(directory):
        for file in files:
            if file.endswith(".sol"):
                filepath = os.path.join(root, file)
                try:
                    with open(filepath, "r") as f:
                        content = f.read()
                        content = remove_comments(content)
                        if "call.value" in content or ".call{value" in content:
                            files_with_call_value.append(filepath)
                        if "block.timestamp " in content or " now " in content:
                            files_with_block_timestamp.append(filepath)
                except Exception as e:
                    print(f"Error reading file {filepath}: {e}")

    return files_with_call_value, files_with_block_timestamp


# Example usage: replace with your actual directory
directory = "/content/smartbugs-wild/contracts"
call_value_files, block_timestamp_files = get_sol_files_with_keywords(directory)

print("Files with 'call.value':", len(call_value_files))
print("Files with 'block.timestamp':", len(block_timestamp_files))

Files with 'call.value': 1605
Files with 'block.timestamp': 7042


In [ ]:
def remove_comments(solidity_code):
    # Regex patterns to match single-line and multi-line comments
    single_line_comment_pattern = r"//.*?(?=\n|$)"
    multi_line_comment_pattern = r"/\*.*?\*/"

    # Remove single-line comments
    code_without_single_line_comments = re.sub(single_line_comment_pattern, '', solidity_code, flags=re.DOTALL)

    # Remove multi-line comments
    cleaned_code = re.sub(multi_line_comment_pattern, '', code_without_single_line_comments, flags=re.DOTALL)

    return cleaned_code.strip()

def find_function(lines, error_line):
    function_pattern = re.compile(r'\bfunction\b')
    modifier_pattern = re.compile(r'\bmodifier\b')
    constructor_pattern = re.compile(r'\bconstructor\b')
    start_line = None

    for i in range(error_line - 1, -1, -1):
        if function_pattern.search(lines[i]) or modifier_pattern.search(lines[i]) or constructor_pattern.search(lines[i]):
            start_line = i
            break

    if start_line is None:
        raise Exception("Function definition not found.")

    end_line = None
    brace_count = 0
    in_function = False

    for i in range(start_line, len(lines)):
        line = lines[i]
        brace_count += line.count('{')
        brace_count -= line.count('}')

        if brace_count > 0:
            in_function = True
        elif brace_count == 0 and in_function:
            end_line = i
            break


    if end_line is None:
        raise Exception("Function end not found.")

    function_code = "".join(lines[start_line:end_line + 1])
    return function_code

In [ ]:
def get_vulnerability_functions(file_path, keyword):
    with open(file_path, "r", encoding='utf-8') as f:
        lines = f.readlines()
        sol_file = "".join(lines)
        sol_file = remove_comments(sol_file)
        rc_lines = sol_file.splitlines(keepends=True)

    functions = []

    for q in range(len(rc_lines)):
        if 'function' in rc_lines[q] or 'modifier' in rc_lines[q] or 'constructor' in rc_lines[q]:
            function = find_function(rc_lines, q + 1)
            if function.count('function') == 1 and function.count('Interface') == 0 and keyword in function:
                functions.append(function.strip())

    output_string = "\n".join(functions)
    return output_string

print(get_vulnerability_functions(block_timestamp_files[100], "block.timestamp"))

function setTime(uint _start, uint _end)
    inState(State.Setup) onlyOwner public 
  {
    require(_start < _end);
    require(_end > block.timestamp);
    startTime = _start;
    endTime = _end;
  }
function ended() public constant returns(bool) {
    return capped() || block.timestamp >= endTime;
  }
function buyTokens(address _beneficiary) inState(State.Active) public payable {
    require(!isDisableEther);
    uint shipAmount = sellTokens(_beneficiary, msg.value, block.timestamp);
    require(shipAmount > 0);
    forwardEther();
  }
function AltCrowdsalePhaseOne (
    address _registry,
    address _token,
    address _extraTokensHolder,
    address _wallet
  )
  BaseAltCrowdsale(
    _registry,
    _token,
    _extraTokensHolder,
    _wallet,

    
    false,

    
    uint(1 ether).div(100000), 

    
    block.timestamp,
    
    1527764400,

    
    2500 ether,
    
    7500 ether
  ) 
  public {
  }
